In [54]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

### Path("knowledge-base") 
Returns a Path object (concretely a PosixPath on Mac/Linux or WindowsPath on Windows). It’s not a string and not a file handle. It’s a path object that represents the location knowledge-base, and it’s relative to whatever directory your script or notebook is running from.

#### Key points:

Creating it does not touch the disk. The folder doesn’t have to exist, and no check happens yet.
It gives you methods for working with that location: .exists(), .is_dir(), .iterdir(), .glob("*.md"), .read_text(), and so on.
It supports / for joining: KNOWLEDGE_BASE_PATH / "notes.md" gives Path("knowledge-base/notes.md").
If you print it, you see knowledge-base. In a notebook cell, the repr is PosixPath('knowledge-base').

In [55]:
load_dotenv(override=True)
MODEL = "gpt-4.1-nano"
DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai = OpenAI()

In [56]:
class Result(BaseModel):
    page_content: str
    metadata: dict

In [57]:
class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text, metadata=metadata)

In [58]:
class Chunks(BaseModel):
    chunks: list[Chunk]

### KNOWLEDGE_BASE_PATH.iterdir() 

KNOWLEDGE_BASE_PATH.iterdir() returns a generator object, not a list. It lazily yields one Path object for each item directly inside that folder, both files and subfolders, in arbitrary order.

Nothing has been read yet. To see the contents, consume the generator.

In [59]:
def fetch_documents():
    """ A self coded version of function similar to LangChain DirectoryLoader """
    documents = [] # List to hold list of documents
    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.glob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type,"source": file.as_posix(), "text": f.read()})
    print(f"Loaded {len(documents)} documents")
    return documents


In [60]:
documents = fetch_documents()

Loaded 76 documents


In [61]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f""" You take a document and you split the document into overlapping chunks for a KnowledgeBase.

    The document is from the shared drive of a company called Insurellm.
    The document is of type: {document["type"]}
    The document has been retrieved from: {document["source"]}

    A chatbot will use these chunks to answer questions about the company.
    You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
    This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
    There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

    For each chunk, you should provide a headline, a summary, and the original text of the chunk.
    Together your chunks should represent the entire document with overlap.

    Here is the document:

    {document["text"]}

    Respond with the chunks.
"""

In [62]:
print(make_prompt(documents[0]))

 You take a document and you split the document into overlapping chunks for a KnowledgeBase.

    The document is from the shared drive of a company called Insurellm.
    The document is of type: products
    The document has been retrieved from: knowledge-base/products/Rellm.md

    A chatbot will use these chunks to answer questions about the company.
    You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
    This document should probably be split into 8 chunks, but you can have more or less as appropriate.
    There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

    For each chunk, you should provide a headline, a summary, and the original text of the chunk.
    Together your chunks should represent the entire document with overlap.

    Here is the document:

    # Prod

In [63]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]


In [64]:
print(make_messages(documents[0]))

[{'role': 'user', 'content': " You take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\n    The document is from the shared drive of a company called Insurellm.\n    The document is of type: products\n    The document has been retrieved from: knowledge-base/products/Rellm.md\n\n    A chatbot will use these chunks to answer questions about the company.\n    You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\n    This document should probably be split into 8 chunks, but you can have more or less as appropriate.\n    There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\n    For each chunk, you should provide a headline, a summary, and the original text of the chunk.\n    Together your chunks should represent the entire document with overl

In [65]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [66]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [67]:
chunks = create_chunks(documents)

100%|██████████| 76/76 [05:45<00:00,  4.54s/it]


In [69]:
print(len(chunks))

389


In [70]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [71]:
create_embeddings(chunks)

Vectorstore created with 389 documents


In [72]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [73]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [74]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()